# 08 — Two Geometries on One Simplex

Close the geometry arc with the punchline that powers the rest of the course: the same simplex carries *two* distinct geometries — information and transport — and they give different answers to "what is the halfway distribution?"

**Prerequisites:** `02/05_the_probability_simplex`, `02/06_fisher_rao_metric`, `02/07_fisher_rao_geodesics`.

**What you'll be able to do**
- Distinguish a "vertical" (information) metric from a "horizontal" (transport) metric.
- Interpolate two distributions the *mixture* way and the *Wasserstein* way.
- Explain why the difference — bimodal blend vs sliding peak — is the reason the course turns to optimal transport.

You have built one full geometry of the simplex: Fisher–Rao (`02/06`–`07`). It is an *information* geometry — it compares masses bin by bin, ignoring what the bins *are*. This notebook reveals a second, completely different geometry hiding on the same simplex, and contrasting the two is the hinge into Movements 3 and 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.geometry.info_geometry import (
    mixture_interpolation, wasserstein_interpolation_1d,
)

np.random.seed(42)
viz.use_course_style()

## 1. A second geometry: optimal transport

Every distance we have used so far — KL, mutual information, Fisher–Rao — is a *"vertical"* metric: it compares masses **bin by bin**, ignoring what the bins are. That is the right view when the bins are unrelated symbols (heads/tails, words in a vocabulary).

But often the bins **have their own geometry** — positions on a line, pixels in an image, neurons on a cortex — and *moving* mass between them has a cost. That is the **transport geometry** (Wasserstein) we will spend Movements 3 and 4 on. It gives a different distance, a different geodesic, and a different answer to "what is the natural path between two distributions?"

To see the contrast, take two Gaussian bumps on a 1-D line, well separated, and interpolate them in **two** geometries.

In [ ]:
support = np.linspace(0.0, 24.0, 200)

def bump(center, sigma=1.2):
    raw = np.exp(-0.5 * ((support - center) / sigma) ** 2)
    return raw / raw.sum()

p_left = bump(center=4.0)
p_right = bump(center=20.0)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(support, p_left, color=viz.SOURCE_COLOR, lw=2, label="$p_0$ centered at $x=4$")
ax.plot(support, p_right, color=viz.TARGET_COLOR, lw=2, label="$p_1$ centered at $x=20$")
ax.set_xlabel("position  $x$")
ax.set_ylabel("probability mass")
ax.set_title("Two distributions on a line — same simplex, two metrics ahead", pad=12)
ax.legend()
plt.show()

**Read the figure.** Two well-separated bumps of the same width. We now ask: what is the "halfway distribution"? The two geometries will give two genuinely different answers.

## 2. Two interpolations — the punchline

For $t \in \{0, 0.25, 0.5, 0.75, 1\}$ we compute:

- the **mixture interpolation** (the *information* answer): $p_t(x) = (1-t)\,p_0(x) + t\,p_1(x)$ — a linear blend of masses, bin by bin;
- the **Wasserstein interpolation** (the *transport* answer): each unit of mass at quantile $u$ in $p_0$ slides to the quantile-$u$ position in $p_1$ (McCann, 1997).

Watch what each one does.

In [ ]:
ts = [0.0, 0.25, 0.5, 0.75, 1.0]
mix_snaps = [mixture_interpolation(p_left, p_right, t) for t in ts]
w2_snaps = [wasserstein_interpolation_1d(p_left, p_right, support, t) for t in ts]

viz.plot_interpolation_panel(
    support,
    ts,
    {"mix": mix_snaps, "w2": w2_snaps},
    titles={
        "mix": "Mixture interpolation — amplitudes morph (bimodal at $t=0.5$)",
        "w2": "Wasserstein interpolation — the peak slides (single mode)",
    },
)
plt.show()

**Read both panels — this is the whole arc.**

- **Top (mixture / information).** At $t = 0.5$ the distribution is **bimodal**: half the mass at $x = 4$, half at $x = 20$, and *nothing in between*. The mixture does not care that there is empty space between the peaks; it blends *amplitudes*. The "halfway distribution" is a mixture, not a displacement.
- **Bottom (Wasserstein / transport).** The peak **slides** smoothly from $x = 4$ to $x = 20$, staying a single bump of nearly the same width. Wasserstein **knows** the bins live on a line and that empty space between two piles costs something to cross.

Same endpoints, same simplex — **two geometries, two different shortest paths**. The Fisher–Rao bow of `02/07` is the curved cousin of the mixture path; both belong to the **information** family. The Wasserstein interpolation belongs to a different family altogether, and that is the entire reason the next nine notebooks build optimal transport, then its quantum lift. *(Cf. Peyré & Cuturi, Fig. 7.4.)*

## 3. Dictionary update

The geometry arc adds the metric and connection rows; their quantum lifts arrive in `02/12` (Bures) and Movements 3–4 (quantum Wasserstein).

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ *(02/09)* |
| **Fisher–Rao metric on the simplex** | **Bures metric on density matrices** *(02/12)* |
| **mixture (m-) vs exponential (e-) connections** | **quantum dual connections** |
| **Wasserstein geometry on distributions** | **quantum Wasserstein on density matrices** *(M4)* |

## Your turn

1. **Bimodality threshold.** Move the right bump toward the left and find the separation at which the mixture midpoint stops being bimodal. How does it compare to $2\sigma$?
2. **Wasserstein stays unimodal.** Confirm the Wasserstein midpoint is a single bump for *any* separation. Why does sliding preserve the shape while blending does not?
3. **Same endpoints, different middle.** For your own pair of bumps, overlay the mixture and Wasserstein midpoints on one axis and describe the difference in a sentence.

## What you built

- You distinguished a "vertical" information metric from a "horizontal" transport metric.
- You interpolated two distributions the mixture way (bimodal blend) and the Wasserstein way (sliding peak).
- You saw, in one figure, why the course turns to optimal transport — and added the geometry rows to the dictionary.

This is the conceptual hinge of the whole course. Everything from here — classical optimal transport (M3) and its quantum lift (M4) — develops the transport geometry you met here. Beautifully done.

## What's next

We close Movement 2 by lifting the information toolbox to density matrices. In `02/09_umegaki_relative_entropy` the quantum KL reappears — and revives the $|+\rangle$ vs $I/2$ seed from `01/06`: it sees the coherence a classical, diagonal-only KL cannot.

## References

- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 7. DOI:10.1561/2200000073
- R. J. McCann, "A convexity principle for interacting gases", *Advances in Mathematics* 128, 153–179 (1997). DOI:10.1006/aima.1997.1634
- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016).

**Previous:** `notebooks/02_InformationAndGeometry/07_fisher_rao_geodesics.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/09_umegaki_relative_entropy.ipynb`